In [2]:
import pandas as pd
import numpy as np
import joblib



In [3]:
df = pd.read_csv('results.csv')
df = df.iloc[45807:]
df.dropna(inplace=True)
df.drop(columns=['tournament','city','country'] , inplace=True)
df =df.sort_values('date').reset_index(drop=True)
df['match_id'] = df.index
df.head(2)

,date,home_team,away_team,home_score,away_score,neutral,match_id
0,2023-01-02,Thailand,Cambodia,3.0,1.0,False,0
1,2023-01-02,Philippines,Indonesia,1.0,2.0,False,1


In [4]:
#home team data
home_df =df[['match_id','date','home_team','home_score','away_score']].copy()
home_df = home_df.rename(columns={'home_team':'team','home_score':'scored','away_score':'conceded'})
home_df['is_home'] = 1
#away team data
away_df =df[['match_id','date','away_team','away_score','home_score']].copy()
away_df = away_df.rename(columns={'away_team':'team','away_score':'scored','home_score':'conceded'})
away_df['is_home'] = 0


In [5]:
team_stats = pd.concat([home_df, away_df]).sort_values(['team','date']).reset_index(drop=True)
team_stats['rolling_scored_5'] = team_stats.groupby('team')['scored'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
team_stats['rolling_conceded_5'] = team_stats.groupby('team')['conceded'].transform(lambda x: x.shift(1).rolling(5,min_periods=1).mean())
team_stats

,match_id,date,team,scored,conceded,is_home,rolling_scored_5,rolling_conceded_5
0,237,2023-06-13,Afghanistan,1.0,6.0,0,NaN,NaN
1,503,2023-09-03,Afghanistan,0.0,0.0,0,1.000000,6.000000
2,542,2023-09-07,Afghanistan,1.0,1.0,0,0.500000,3.000000
3,627,2023-09-12,Afghanistan,1.0,2.0,0,0.666667,2.333333
4,705,2023-10-12,Afghanistan,1.0,0.0,1,0.750000,2.250000
...,...,...,...,...,...,...,...,...
7247,3467,2026-05-30,Zimbabwe,1.0,0.0,1,1.400000,1.600000
7248,442,2023-07-09,Åland Islands,0.0,2.0,1,NaN,NaN
7249,452,2023-07-10,Åland Islands,1.0,4.0,0,0.000000,2.000000
7250,456,2023-07-11,Åland Islands,0.0,2.0,1,0.500000,3.000000


In [6]:
home_featuers = team_stats[team_stats['is_home'] == 1][['match_id','rolling_scored_5','rolling_conceded_5']].copy()
home_featuers.rename(columns={'rolling_scored_5': 'home_rolling_scored_5',
    'rolling_conceded_5': 'home_rolling_conceded_5'
}, inplace=True)
#away
away_featuers = team_stats[team_stats['is_home'] == 0][['match_id' ,'rolling_scored_5','rolling_conceded_5']].copy()
away_featuers.rename(columns={'rolling_scored_5': 'away_rolling_scored_5',
    'rolling_conceded_5': 'away_rolling_conceded_5'
}, inplace=True)

In [7]:
#merg
df =df.merge(home_featuers , on='match_id' ,how='left')
df =df.merge(away_featuers, on='match_id' ,how='left')
df.tail(3)

,date,home_team,away_team,home_score,away_score,neutral,match_id,home_rolling_scored_5,home_rolling_conceded_5,away_rolling_scored_5,away_rolling_conceded_5
3623,2026-06-18,Czech Republic,South Africa,1.0,1.0,True,3623,2.4,1.6,0.6,1.0
3624,2026-06-18,Mexico,South Korea,1.0,0.0,False,3624,2.2,0.4,1.6,1.2
3625,2026-06-18,Canada,Qatar,6.0,0.0,False,3625,1.2,0.8,0.4,1.2


In [8]:
rank = pd.read_csv(r"fifa_ranking_2026-06-08.csv")
rank.drop(columns=['team_code','association','previous_rank','points','previous_points','rated_matches'] ,inplace=True)
rank.head(5)

,team,rank
0,Argentina,1
1,Spain,2
2,France,3
3,England,4
4,Portugal,5


In [9]:
home_rank_df = rank[['team','rank']].copy()
home_rank_df.rename(columns={
    'team':'home_team',
    'rank':'home_team_rank'
},inplace=True)
#away_rank
away_rank_df = rank[['team','rank']].copy()
away_rank_df.rename(columns={
    'team':'away_team',
    'rank':'away_team_rank'
},inplace=True)
df = df.merge(home_rank_df , on='home_team' ,how='left')
df = df.merge(away_rank_df , on='away_team' ,how='left')


In [10]:
df

,date,home_team,away_team,home_score,away_score,neutral,match_id,home_rolling_scored_5,home_rolling_conceded_5,away_rolling_scored_5,away_rolling_conceded_5,home_team_rank,away_team_rank
0,2023-01-02,Thailand,Cambodia,3.0,1.0,False,0,NaN,NaN,NaN,NaN,94.0,176.0
1,2023-01-02,Philippines,Indonesia,1.0,2.0,False,1,NaN,NaN,NaN,NaN,135.0,119.0
2,2023-01-03,Vietnam,Myanmar,3.0,0.0,False,2,NaN,NaN,NaN,NaN,99.0,158.0
3,2023-01-03,Malaysia,Singapore,4.0,1.0,False,3,NaN,NaN,NaN,NaN,137.0,149.0
4,2023-01-06,Iraq,Oman,0.0,0.0,False,4,NaN,NaN,NaN,NaN,56.0,79.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...
3621,2026-06-17,Ghana,Panama,1.0,0.0,True,3621,0.6,2.2,2.0,2.2,73.0,34.0
3622,2026-06-18,Switzerland,Bosnia and Herzegovina,4.0,1.0,True,3622,1.8,1.4,0.8,0.8,19.0,64.0
3623,2026-06-18,Czech Republic,South Africa,1.0,1.0,True,3623,2.4,1.6,0.6,1.0,NaN,60.0
3624,2026-06-18,Mexico,South Korea,1.0,0.0,False,3624,2.2,0.4,1.6,1.2,14.0,NaN


In [17]:
cond = [
    (df['home_score'] > df['away_score']),
    (df['away_score'] == df['home_score']),
    (df['home_score'] < df['away_score'])
]
choices = [2,1,0]
df['target'] = np.select(cond , choices ,default=-1)
df_clean = df.dropna().reset_index(drop=True)
df_clean['neutral']=df_clean['neutral'].astype(int)
df_clean['rank_diff'] = df_clean['home_team_rank'] - df_clean['away_team_rank']
df_clean.to_csv('data.csv',index=False)
df_clean

,date,home_team,away_team,home_score,away_score,neutral,match_id,home_rolling_scored_5,home_rolling_conceded_5,away_rolling_scored_5,away_rolling_conceded_5,home_team_rank,away_team_rank,target,rank_diff
0,2023-01-06,Indonesia,Vietnam,0.0,0.0,0,6,2.0,1.0,3.0,0.0,119.0,99.0,1,20.0
1,2023-01-07,Malaysia,Thailand,1.0,0.0,0,9,4.0,1.0,3.0,1.0,137.0,94.0,2,43.0
2,2023-01-09,Vietnam,Indonesia,2.0,0.0,0,11,1.5,0.0,1.0,0.5,99.0,119.0,2,-20.0
3,2023-01-09,Iraq,Saudi Arabia,2.0,0.0,0,12,0.0,0.0,2.0,0.0,56.0,61.0,2,-5.0
4,2023-01-09,Oman,Yemen,3.0,2.0,1,14,0.0,0.0,0.0,2.0,79.0,145.0,2,-66.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2735,2026-06-17,Uzbekistan,Colombia,1.0,3.0,1,3619,1.2,1.4,2.0,1.2,50.0,13.0,0,37.0
2736,2026-06-17,England,Croatia,4.0,2.0,1,3620,1.4,0.4,1.6,1.8,4.0,11.0,2,-7.0
2737,2026-06-17,Ghana,Panama,1.0,0.0,1,3621,0.6,2.2,2.0,2.2,73.0,34.0,2,39.0
2738,2026-06-18,Switzerland,Bosnia and Herzegovina,4.0,1.0,1,3622,1.8,1.4,0.8,0.8,19.0,64.0,2,-45.0


In [12]:
features = [
    'home_rolling_scored_5', 'home_rolling_conceded_5',
    'away_rolling_scored_5', 'away_rolling_conceded_5',
    'home_team_rank', 'away_team_rank', 
    'neutral' , 'rank_diff'
]

In [13]:
X = df_clean[features]
y = df_clean['target']
y.shape , X.shape

((2740,), (2740, 8))

In [14]:
np.savez('preprocessed_data.npz',X=X,y=y)